In [75]:
import torch
import pandas as pd

In [76]:
df = pd.read_csv("dataset_j.csv")
df.sample(2)

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
8532,8533,"Director, Sales","CA, ON, Greater Toronto Area",NaN,NaN,MMR Inc is one of Canada’s highest accomplishe...,Our client is a provider of high quality busi...,The qualified candidate is someone who is a te...,NaN,0,1,1,Full-time,Director,NaN,Business Supplies and Equipment,Management,0
17284,17285,Senior Full-stack Developer (Pair program in R...,"US, CA, San Francisco",Engineering,NaN,SocialChorus® powers tens of thousands of bran...,Tech StackFull-stack coding in Ruby and JavaSc...,3+ years of full-time Ruby/Rails and JavaScrip...,Free code FridayMedical and dental Paid vacati...,0,1,1,Full-time,Mid-Senior level,NaN,Computer Software,Engineering,0


Exploration of dataset

In [77]:
df.sample(2)

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
14840,14841,Consumer Public Relations Intern (DC),"US, DC, Washington",NaN,NaN,We are a PR and social media agency that goes ...,DBC PR + Social Media is an original PR agency...,Ideal candidates will have an interest in publ...,NaN,0,1,0,Part-time,Internship,NaN,Public Relations and Communications,Public Relations,0
6667,6668,Physical Therapist Outpatient,"US, VA, Newport News",NaN,NaN,Supply chain management company with emphasis ...,Outpatient clinic is interested in a Physical ...,Physical Therapy Degree,NaN,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


In [78]:
print("No. of columns: ", df.columns.nunique())
print("Name of columns: ", df.columns)

No. of columns:  18
Name of columns:  Index(['job_id', 'title', 'location', 'department', 'salary_range',
       'company_profile', 'description', 'requirements', 'benefits',
       'telecommuting', 'has_company_logo', 'has_questions', 'employment_type',
       'required_experience', 'required_education', 'industry', 'function',
       'fraudulent'],
      dtype='object')


In [79]:
print(f"Shape of data (rows x columns): {df.shape}")

Shape of data (rows x columns): (17880, 18)


In [80]:
df.isna().sum()

,0
job_id,0
title,0
location,346
department,11547
salary_range,15012
company_profile,3308
description,1
requirements,2696
benefits,7212
telecommuting,0


Handling Missing Values

In [81]:
df.fillna('', inplace=True)

In [82]:
df = df.drop(columns=['job_id', 'salary_range'])

In [83]:
df['industry'].nunique()

132

In [84]:
df.columns

Index(['title', 'location', 'department', 'company_profile', 'description',
       'requirements', 'benefits', 'telecommuting', 'has_company_logo',
       'has_questions', 'employment_type', 'required_experience',
       'required_education', 'industry', 'function', 'fraudulent'],
      dtype='object')

In [85]:
text_cols = ['title', 'location', 'department', 'industry', 'company_profile', 'description',
       'requirements', 'benefits', 'employment_type', 'required_experience',
       'required_education', 'function']

In [86]:
df["text"] = df[text_cols].agg(" ".join, axis=1)

In [87]:
df['label'] = df['fraudulent'].map({0:0, 1:1})

In [88]:
df = df.drop_duplicates(subset=["text", "label"])

In [89]:
X_df = df['text']
X_extra= df[['telecommuting', 'has_company_logo', 'has_questions']]

In [90]:
y_df = df['label']

Train Test Split

In [91]:
from sklearn.model_selection import train_test_split

Train_text, Test_text, Train_extra, Test_extra, Train_labels, Test_labels = train_test_split(
    X_df, X_extra, y_df, test_size=0.2, stratify=y_df, random_state=42
)

Tokenizer with NLTK pipeline

In [92]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [93]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [94]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [95]:
def tokenize(text):
  text=text.lower()
  tokens=word_tokenize(text)

  tokens = [re.sub(r'\W+', '', t)for t in tokens]
  tokens = [t for t in tokens if t!='']

  tokens = [t for t in tokens if t not in stop_words]

  tokens = [lemmatizer.lemmatize(t) for t in tokens]

  return tokens

Vocabulary Building

In [96]:
from collections import Counter

vocab_size = 20000
counter = Counter()

for text in Train_text:
    counter.update(tokenize(text))

vocab = {word: i+2 for i, (word, _) in enumerate(counter.most_common(vocab_size))}

vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

Text->Numbers

In [97]:
def encode(text):
    return [vocab.get(word, 1) for word in tokenize(text)]

In [98]:
import torch
from torch.nn.utils.rnn import pad_sequence

def prepare_sequences(texts):
    sequences = [torch.tensor(encode(t)) for t in texts]
    padded = pad_sequence(sequences, batch_first=True, padding_value=0)
    return padded

Full Inputs

In [99]:
X_train_text = prepare_sequences(Train_text)
X_val_text   = prepare_sequences(Test_text)

X_train_extra = torch.tensor(Train_extra.values, dtype=torch.float32)
X_val_extra   = torch.tensor(Test_extra.values, dtype=torch.float32)

y_train_tensor = torch.tensor(Train_labels.values, dtype=torch.float32)
y_val_tensor   = torch.tensor(Test_labels.values, dtype=torch.float32)

In [100]:
import torch
import torch.nn as nn

In [101]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, extra_feat_dim):
        super(LSTMClassifier, self).__init__()

        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # LSTM
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        # Extra features branch
        self.extra_fc = nn.Linear(extra_feat_dim, 32)

        # Final classifier
        self.fc = nn.Linear(hidden_dim*2 + 32, 1)

        self.dropout = nn.Dropout(0.3)

    def forward(self, text, extra):
        # text: (batch_size, seq_len)
        # extra: (batch_size, extra_feat_dim)

        embedded = self.dropout(self.embedding(text))
        # (batch_size, seq_len, embed_dim)

        lstm_out, (hidden, _) = self.lstm(embedded)

        # Bidirectional fix
        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]

        text_features = torch.cat((forward_hidden, backward_hidden), dim=1)
        # (batch_size, hidden_dim)

        text_features = self.dropout(text_features)

        # Process extra features
        extra_features = torch.relu(self.extra_fc(extra))

        # Combine
        combined = torch.cat((text_features, extra_features), dim=1)

        output = self.fc(combined)  # NO sigmoid here

        return output

Initializing Model

In [102]:
vocab_size = len(vocab)
embed_dim = 128
hidden_dim = 128
extra_feat_dim = X_train_extra.shape[1]

model = LSTMClassifier(vocab_size, embed_dim, hidden_dim, extra_feat_dim)

Defining loss and optimizer

In [103]:
from sklearn.utils.class_weight import compute_class_weight
import torch
import numpy as np

In [104]:
print(df["fraudulent"].value_counts())

fraudulent
0    16726
1      855
Name: count, dtype: int64


In [105]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=Train_labels   # ✅ pandas series, NOT tensor
)

pos_weight = torch.tensor(weights[1]*1.5, dtype=torch.float32)

In [106]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [107]:
y_train_tensor = y_train_tensor.view(-1, 1)
y_val_tensor   = y_val_tensor.view(-1, 1)

In [108]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

X_train_text = X_train_text.to(device)
X_val_text   = X_val_text.to(device)

X_train_extra = X_train_extra.to(device)
X_val_extra   = X_val_extra.to(device)

y_train_tensor = y_train_tensor.to(device)
y_val_tensor   = y_val_tensor.to(device)

In [109]:
class JobDataset(torch.utils.data.Dataset):
    def __init__(self, texts, extras, labels):
        self.texts = texts
        self.extras = extras
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.texts[idx], self.extras[idx], self.labels[idx]

In [110]:
from torch.utils.data import DataLoader

batch_size = 32  # 🔥 try 16 if still OOM

train_dataset = JobDataset(X_train_text, X_train_extra, y_train_tensor)
val_dataset   = JobDataset(X_val_text, X_val_extra, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size)

In [111]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for text_batch, extra_batch, label_batch in train_loader:

        text_batch = text_batch.to(device)
        extra_batch = extra_batch.to(device)
        label_batch = label_batch.to(device)

        optimizer.zero_grad()

        outputs = model(text_batch, extra_batch)
        loss = criterion(outputs, label_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # ---- VALIDATION ----
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for text_batch, extra_batch, label_batch in val_loader:
            text_batch = text_batch.to(device)
            extra_batch = extra_batch.to(device)
            label_batch = label_batch.to(device)

            outputs = model(text_batch, extra_batch)
            loss = criterion(outputs, label_batch)

            val_loss += loss.item()

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {total_loss:.4f} | Val Loss: {val_loss:.4f}")

Epoch 1
Train Loss: 332.7570 | Val Loss: 41.4250
Epoch 2
Train Loss: 160.0114 | Val Loss: 34.0246
Epoch 3
Train Loss: 103.6676 | Val Loss: 29.2693
Epoch 4
Train Loss: 61.1704 | Val Loss: 31.3264
Epoch 5
Train Loss: 47.9991 | Val Loss: 39.4018
Epoch 6
Train Loss: 79.2658 | Val Loss: 38.4717
Epoch 7
Train Loss: 63.5737 | Val Loss: 38.3945
Epoch 8
Train Loss: 38.6981 | Val Loss: 46.0768
Epoch 9
Train Loss: 26.6158 | Val Loss: 46.7604
Epoch 10
Train Loss: 18.9292 | Val Loss: 58.4827


In [112]:
model.eval()

all_probs = []
all_labels = []

with torch.no_grad():
    for text_batch, extra_batch, label_batch in val_loader:

        text_batch = text_batch.to(device)
        extra_batch = extra_batch.to(device)

        outputs = model(text_batch, extra_batch)

        probs = torch.sigmoid(outputs)

        all_probs.append(probs.cpu())
        all_labels.append(label_batch)  # already on CPU usually

In [113]:
import torch

all_probs = torch.cat(all_probs)
all_labels = torch.cat(all_labels)

In [114]:
from sklearn.metrics import classification_report

for t in [0.5, 0.6, 0.7, 0.8]:
    preds = (all_probs >= t).float()

    print(f"\nThreshold: {t}")
    print(classification_report(
        all_labels.cpu().numpy(),
        preds.cpu().numpy()
    ))


Threshold: 0.5
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99      3346
         1.0       0.84      0.84      0.84       171

    accuracy                           0.98      3517
   macro avg       0.91      0.92      0.92      3517
weighted avg       0.98      0.98      0.98      3517


Threshold: 0.6
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99      3346
         1.0       0.85      0.84      0.84       171

    accuracy                           0.98      3517
   macro avg       0.92      0.91      0.92      3517
weighted avg       0.98      0.98      0.98      3517


Threshold: 0.7
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99      3346
         1.0       0.86      0.82      0.84       171

    accuracy                           0.98      3517
   macro avg       0.93      0.91      0.92      3517
weighted avg       0.98   

In [115]:
print("fraudulent" in text_cols)  # should be False

False


In [116]:
len(set(Train_text).intersection(set(Test_text)))

0

In [117]:
import numpy as np
from sklearn.metrics import f1_score

y_true = all_labels.cpu().numpy()
y_probs = all_probs.cpu().numpy()

thresholds = np.arange(0.1, 0.9, 0.05)

best_t = 0
best_f1 = 0

for t in thresholds:
    preds = (y_probs >= t).astype(int)
    f1 = f1_score(y_true, preds)

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print(f"Best Threshold: {best_t}")
print(f"Best F1: {best_f1}")

Best Threshold: 0.7500000000000002
Best F1: 0.8493975903614458


In [118]:
THRESHOLD = best_t

In [128]:
import torch

# Suppose you have:
# model -> your trained LSTMClassifier
# vocab -> the vocabulary dictionary
# best_threshold -> the threshold you chose based on validation

save_path = "job_classifier_v1.pth"

torch.save({
    "model_state_dict": model.state_dict(),
    "vocab": vocab,
    "threshold": THRESHOLD
}, save_path)

print(f"✅ Model, vocab, and threshold saved to {save_path}")

✅ Model, vocab, and threshold saved to job_classifier_v1.pth


Loading Model

In [130]:
import torch

# 1. Recreate the same model architecture
model = LSTMClassifier(
    vocab_size=len(vocab),  # placeholder, will overwrite after loading
    embed_dim=128,
    hidden_dim=128,
    extra_feat_dim=X_train_extra.shape[1]  # same as during training
)

# 2. Load checkpoint
checkpoint = torch.load("job_classifier_v1.pth", weights_only=False)

# 3. Load weights
model.load_state_dict(checkpoint["model_state_dict"])

# 4. Load vocab and threshold
vocab = checkpoint["vocab"]
THRESHOLD = checkpoint["threshold"]

# 5. Set model to evaluation mode
model.eval()

print("✅ Model loaded and ready for inference")
print(f"Loaded threshold: {THRESHOLD}")

✅ Model loaded and ready for inference
Loaded threshold: 0.7500000000000002


In [145]:
def predict(text, extra_features):
    """
    Predict if a job is real or fake using the trained LSTM model.

    Args:
        text (str): Job posting text
        extra_features (list or tensor): [telecommuting, has_company_logo, has_questions]

    Returns:
        dict: Prediction info including probabilities, predicted class, and human-readable label
    """
    # 1. Encode text
    encoded = [vocab.get(word, vocab.get("<UNK>", 1)) for word in tokenize(text)]
    text_tensor = torch.tensor([encoded])  # batch size 1
    text_tensor = torch.nn.utils.rnn.pad_sequence(text_tensor, batch_first=True, padding_value=0)

    # 2. Extra features
    extra_tensor = torch.tensor([extra_features], dtype=torch.float32)

    # 3. Forward pass
    with torch.no_grad():
        output = model(text_tensor, extra_tensor)
        prob_fake = torch.sigmoid(output).item()  # probability for class 1 (Fake)
        prob_real = 1 - prob_fake
        pred_class = int(prob_fake >= THRESHOLD)

    # 4. Map class to human-readable label
    label_map = {0: "Real Job Posting", 1: "Fake Job Posting"}
    label = label_map[pred_class]

    return {
        "prediction": pred_class,
        "label": label,
        "prob_real": prob_real,
        "prob_fake": prob_fake
    }

# Example usage
example_text = "Software Engineer needed at tech company. earn 50000 daily"
example_extra = [0, 1, 0]  # telecommuting, has_company_logo, has_questions

result = predict(example_text, example_extra)

# Determine risk level based on fake probability
if result['prediction'] == 1:
    risk = "⚠️ High risk of fake posting"
else:
    if result['prob_fake'] > 0.5:
        risk = "ℹ️ Some caution advised"
    else:
        risk = "✅ Low risk of fake posting"

# Print formatted output
print("🔹 Job Posting Analysis")
print(f"Text: {example_text}")
print(f"Prediction: {result['label']} (class {result['prediction']})")
print(f"Confidence → Real: {result['prob_real']*100:.2f}%, Fake: {result['prob_fake']*100:.2f}%")
print(f"Risk Assessment: {risk}")

🔹 Job Posting Analysis
Text: Software Engineer needed at tech company. earn 50000 daily
Prediction: Real Job Posting (class 0)
Confidence → Real: 33.29%, Fake: 66.71%
Risk Assessment: ℹ️ Some caution advised
